# Whisper diagnosis

`whisper-large-v3` and `-turbo` both produce roughly half the reference
words, with ~50% of the error being deletions. Beam size and conditioning
make no difference. Sarvam reaches ratio 0.96 on the same audio, so the
reference is right and the audio is transcribable.

One clip shows an outright repetition loop — 550 s of
`प्रशिक्षक मनों नेम्नूक जालेला है` repeated — but short clips also
under-generate without much repetition, so there may be two faults.

These cells read Whisper's own per-segment metadata, which our stored
transcripts discard, and test the settings that control what it throws away.

In [ ]:
# --- sync + imports --------------------------------------------------------
import subprocess, sys, shutil
from pathlib import Path

CODE = Path("/kaggle/working/sarvam-assignment")
if (CODE/".git").exists():
    subprocess.run(["git","-C",str(CODE),"fetch","-q","origin","main"], check=True)
    subprocess.run(["git","-C",str(CODE),"reset","-q","--hard","origin/main"], check=True)
else:
    subprocess.run(["git","clone","-q",
                    "https://github.com/ParvGoyal08/MultilingualASR.git", str(CODE)], check=True)
print("code @", subprocess.run(["git","-C",str(CODE),"log","-1","--format=%h %s"],
                               capture_output=True, text=True).stdout.strip())
for m in [k for k in list(sys.modules) if k=="sarvam_diar" or k.startswith("sarvam_diar.")]:
    del sys.modules[m]
sys.path.insert(0, str(CODE))

from sarvam_diar.config import Config
from sarvam_diar import asr, data, reference
ROOT = "/kaggle/working/sarvam_diarization"
cfg = Config.create(root=ROOT, work_dir=f"{ROOT}/tmp")
CLIPS = {c.clip_id: c for c in data.parse_ground_truth(data.load_segments_csv(cfg))}
print("audio files:", len(list(Path(cfg.audio_dir).glob("*.wav"))))


## 1 — raw segments and their filter metadata

In [ ]:
# --- RAW Whisper output, with the metadata that decides what gets kept ------
# faster-whisper filters and retries segments using no_speech_prob,
# avg_logprob and compression_ratio, and none of that survives into our stored
# transcripts. Reading it directly is the only way to tell whether Whisper is
# DISCARDING audio, producing SHORT segments, or looping.
from faster_whisper import WhisperModel
import numpy as np

MODEL = "large-v3-turbo"
CLIP  = "0AEEA8NyVwY__11_609"          # 598 s, known bad: ratio 0.18
m = asr._whisper_model(cfg, MODEL)
lang, prob = asr.detect_language(cfg, cfg.wav_path(CLIP))
print(f"language: {lang} (p={prob:.2f})")

segs, info = m.transcribe(
    str(cfg.wav_path(CLIP)), language=lang, beam_size=5,
    word_timestamps=False, vad_filter=False,
)
rows = list(segs)
ref = reference.build_reference(CLIPS[CLIP])
ref_w = sum(len(reference.tokenize(u.text_norm)) for u in ref.utterances)
hyp_w = sum(len((s.text or "").split()) for s in rows)
print(f"\nsegments: {len(rows)}   words: {hyp_w}   reference words: {ref_w}"
      f"   ratio {hyp_w/ref_w:.2f}")
covered = sum(s.end - s.start for s in rows)
print(f"segment time covered: {covered:.0f}s of {ref.uem[1]:.0f}s "
      f"({covered/ref.uem[1]:.0%})")

print(f"\n{'start':>7}{'end':>8}{'w':>4}{'nospeech':>10}{'logprob':>9}{'compr':>7}{'temp':>6}  text")
for s in rows[:30]:
    print(f"{s.start:>7.1f}{s.end:>8.1f}{len((s.text or '').split()):>4}"
          f"{s.no_speech_prob:>10.3f}{s.avg_logprob:>9.2f}{s.compression_ratio:>7.2f}"
          f"{getattr(s,'temperature',float('nan')):>6.1f}  {(s.text or '')[:52]}")

# gaps between consecutive segments -- audio Whisper emitted nothing for
gaps = [(b.start - a.end) for a, b in zip(rows, rows[1:]) if b.start - a.end > 1.0]
print(f"\ngaps > 1s between segments: {len(gaps)}, totalling {sum(gaps):.0f}s")
print(f"  -> if this is large, Whisper is SKIPPING audio, not just decoding it briefly")


## 2 — do the discard thresholds explain it?

In [ ]:
# --- does disabling the discard thresholds recover the words? ---------------
# no_speech_threshold: drop a window whose no-speech probability exceeds it.
# log_prob_threshold / compression_ratio_threshold: trigger temperature
# fallback, and on repeated failure the segment is dropped as non-speech.
# Turning each off separates "Whisper never decoded it" from "Whisper decoded
# it and threw it away".
CONFIGS = [
    ("current (nst 0.6)",            dict()),
    ("no_speech_threshold=None",     dict(no_speech_threshold=None)),
    ("+ unconditioned",              dict(no_speech_threshold=None,
                                          condition_on_previous_text=False)),
    ("+ single temperature 0.0",     dict(no_speech_threshold=None,
                                          condition_on_previous_text=False,
                                          temperature=[0.0])),
]
print(f"{'config':<30}{'segs':>6}{'words':>7}{'ratio':>7}{'covered':>9}{'top5gram':>10}")
import collections
for label, kw in CONFIGS:
    segs, _ = m.transcribe(str(cfg.wav_path(CLIP)), language=lang, beam_size=5,
                           word_timestamps=False, vad_filter=False, **kw)
    rr = list(segs)
    toks = [t for s in rr for t in (s.text or "").split()]
    g = collections.Counter(tuple(toks[i:i+5]) for i in range(max(0, len(toks)-4)))
    rep = (g.most_common(1)[0][1]*5/len(toks)) if toks else 0
    cov = sum(s.end - s.start for s in rr)
    print(f"{label:<30}{len(rr):>6}{len(toks):>7}{len(toks)/ref_w:>7.2f}"
          f"{cov/ref.uem[1]:>8.0%}{rep:>10.0%}")


## 3 — per-segment, the way Sarvam is run

In [ ]:
# --- per-segment: the strategy Sarvam uses, which removes Whisper's choice ---
from sarvam_diar import diarization
turns = asr.merge_same_speaker(diarization.load_hypothesis(cfg, "reverb-v2", CLIP), 1.0)
segs, meta = asr.transcribe_segments(cfg, f"whisper-{MODEL}", cfg.wav_path(CLIP), turns)
hw = [t for s in segs
      for t in reference.normalize_text(s["text"], strip_gloss=False).split()]
print(f"per-segment over {len(turns)} reverb-v2 turns")
print(f"  words {len(hw)}  reference {ref_w}  ratio {len(hw)/ref_w:.2f}")
print(f"  skipped as too short: {meta.get('n_skipped_short')}")
print(f"\nfirst 6 segments:")
for s in segs[:6]:
    print(f"  {s['start']:7.1f}-{s['end']:7.1f}  {s['speaker']:<12} {s['text'][:60]}")
